# Advanced Analytics: Predictive Model (Regression)
Group 6.1 Labanza and Lontoc. Target: 2021 Inequality in Education. Features: HDI rank, Human Development Group, UNDP region. Previous-year inequality is excluded (r = 0.997, leakage). Baseline: group mean. Models: Linear Regression, Random Forest. Validation: repeated 5-fold CV and an 80/20 hold-out. Priority countries: actual minus out-of-fold predicted above 1 CV-RMSE.

In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import RepeatedKFold, train_test_split

TARGET_YEAR=2021
HDG_ORDER=["Low","Medium","High","Very High"]
REGION_NAMES={"SSA":"Sub-Saharan Africa","SA":"South Asia","AS":"Arab States","LAC":"Latin America & Caribbean","EAP":"East Asia & Pacific","ECA":"Europe & Central Asia","Unclassified":"Unclassified"}
BASELINE,LINEAR,FOREST="Baseline (group mean)","Linear Regression","Random Forest"
def r(x,n=2): return None if x is None or (isinstance(x,float) and np.isnan(x)) else round(float(x),n)

In [ ]:
# In Colab: from google.colab import files; files.upload()  (choose df_long.csv)
long_df=pd.read_csv("df_long.csv")
long_df["HDG"]=long_df["Human Development Groups"].fillna("Unclassified")
long_df["Region_Code"]=long_df["UNDP Developing Regions"].fillna("Unclassified")
long_df["Region"]=long_df["Region_Code"].map(REGION_NAMES)
print(long_df.shape)

In [ ]:
def metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    e = y_true - y_pred
    ss = float(np.sum((y_true - y_true.mean()) ** 2))
    return float(np.sqrt(np.mean(e**2))), float(np.mean(np.abs(e))), (1 - float(np.sum(e**2)) / ss if ss > 0 else float("nan"))


def fit_predict(name, xtr, ytr, gtr, xte, gte):
    if name == BASELINE:
        means, overall = ytr.groupby(gtr).mean(), ytr.mean()
        f = lambda g: g.map(means).fillna(overall).to_numpy()
        return f(gtr), f(gte)
    model = LinearRegression() if name == LINEAR else RandomForestRegressor(
        n_estimators=200, min_samples_leaf=3, random_state=42, n_jobs=-1)
    model.fit(xtr, ytr)
    return model.predict(xtr), model.predict(xte)


d = (long_df[long_df["Year"] == TARGET_YEAR]
     .dropna(subset=["Inequality", "HDI Rank (2021)", "Human Development Groups"]).reset_index(drop=True))
x = pd.DataFrame({
    "HDI_Rank": d["HDI Rank (2021)"].astype(float),
    "HDG_Code": d["Human Development Groups"].map({g: i for i, g in enumerate(HDG_ORDER)}).astype(float),
})
x = pd.concat([x, pd.get_dummies(d["Region_Code"], prefix="Region", dtype=int)], axis=1)
y = d["Inequality"].astype(float)
grp = d["Human Development Groups"].astype(str)
names = [BASELINE, LINEAR, FOREST]

itr, ite = train_test_split(np.arange(len(d)), test_size=0.2, random_state=42)
holdout = []
for n in names:
    ptr, pte = fit_predict(n, x.iloc[itr], y.iloc[itr], grp.iloc[itr], x.iloc[ite], grp.iloc[ite])
    rtr, _, r2tr = metrics(y.iloc[itr], ptr)
    rte, mte, r2te = metrics(y.iloc[ite], pte)
    holdout.append({"model": n, "train_rmse": r(rtr,3), "test_rmse": r(rte, 3), "test_mae": r(mte, 3),
                    "train_r2": r(r2tr, 3), "test_r2": r(r2te, 3)})

sc = {n: {"rmse": [], "mae": [], "r2": []} for n in names}
oof_sum = {n: np.zeros(len(d)) for n in names}
oof_cnt = np.zeros(len(d))
for tr, te in RepeatedKFold(n_splits=5, n_repeats=5, random_state=42).split(x):
    oof_cnt[te] += 1
    for n in names:
        _, p = fit_predict(n, x.iloc[tr], y.iloc[tr], grp.iloc[tr], x.iloc[te], grp.iloc[te])
        a, b, c = metrics(y.iloc[te], p)
        sc[n]["rmse"].append(a); sc[n]["mae"].append(b); sc[n]["r2"].append(c)
        oof_sum[n][te] += p
cv = [{"model": n, "cv_rmse": r(np.mean(sc[n]["rmse"]), 3), "rmse_std": r(np.std(sc[n]["rmse"]), 3),
       "cv_mae": r(np.mean(sc[n]["mae"]), 3), "cv_r2": r(np.mean(sc[n]["r2"]), 3)} for n in names]
best = sorted([c for c in cv if c["model"] != BASELINE], key=lambda c: c["cv_rmse"])[0]
oof = oof_sum[best["model"]] / oof_cnt

results = []
for i in range(len(d)):
    results.append({
        "country": d.loc[i, "Country"], "iso": d.loc[i, "ISO3"], "hdg": d.loc[i, "HDG"], "region": d.loc[i, "Region"],
        "hdi": int(d.loc[i, "HDI Rank (2021)"]), "actual": r(y[i]), "pred": r(oof[i]), "gap": r(y[i] - oof[i]),
    })
rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=42, n_jobs=-1).fit(x, y)
imp = sorted(({"feature": f, "importance": r(v, 4)} for f, v in zip(x.columns, rf.feature_importances_)),
             key=lambda t: t["importance"])
lin = LinearRegression().fit(x, y)
model_out = {
    "target_year": TARGET_YEAR, "n": int(len(d)), "best": best["model"], "best_rmse": best["cv_rmse"],
    "cv": cv, "holdout": holdout, "importance": imp, "results": results,
    "features": list(x.columns),
}
base_cv = next(c for c in cv if c["model"] == BASELINE)
n_prio = sum(1 for t in results if t["gap"] > best["cv_rmse"])
print("CV:", cv)
print("Best:", best["model"], "| priority countries at 1.0 RMSE:", n_prio)



In [ ]:
print(pd.DataFrame(cv)); print(pd.DataFrame(holdout))
res=pd.DataFrame(results); thr=best["cv_rmse"]
print("Priority countries (actual - predicted > 1 RMSE):")
res[res.gap>thr].sort_values("gap",ascending=False)